# Day 1.5 — Build the Agent Loop Manually

One hardcoded tool interaction cannot handle an unknown number of steps. We now build the mechanism that makes this application agentic:

```text
Model → decide → tool → observation → model → ... → final answer
```

The loop is controlled by Python. It has a step limit and explicit failure status.

## Before you begin

### Learning outcomes

Trace repeated model/tool turns and prove the host step limit stops execution.

Architecture reference: [D04](../../diagrams/source/day_01.md).

### Expected observation

Every tool result is appended before the next model call; forced looping ends at max_steps.


## Concept briefing

## Why the application owns termination

After one tool result, the model may ask for another tool or return a final answer. That
creates a loop whose length is not known in advance. It is tempting to write "stop when
finished" in the system message and trust the model. That is not an execution limit. A
confused model can repeat the same request, alternate between tools, or continue refining
an already adequate answer. Each turn consumes time, tokens and money.

Host code therefore enforces a maximum number of steps. Reaching the limit is not the
same as crashing. A good runtime returns a visible status such as `max_steps` together
with the partial trace. Reporting incomplete work honestly is safer than pretending the
run completed.

## Error compounding

Multi-step systems amplify small error rates. Suppose, only for illustration, that each
model decision has a 95% chance of being acceptable and that errors are independent. The
chance that ten decisions are all acceptable is:

```text
0.95 ^ 10 = approximately 0.60
```

The independence assumption is simplistic, but the lesson is useful: a system with many
model decisions can be much less reliable than any single impressive response suggests.
This motivates bounded loops, deterministic validation, fewer calls, clear tools and
evaluation of complete trajectories rather than isolated answers.


## Learning objectives

Trace a multi-step agent run, explain reason/action/observation in application terms, and show why termination and validation belong outside the model.

In [ ]:
import os
import sys
from pathlib import Path

# Locate the Day 1 project whether the notebook starts from the repository root or notebooks folder.
here = Path.cwd().resolve()
candidates = [here, here / "day_01_model_tools_agent", here.parent]
project_root = next(path for path in candidates if (path / "src" / "research_agent").exists())
sys.path.insert(0, str(project_root / "src"))

from research_agent.agent import AgentRunner
from research_agent.providers import OpenRouterProvider
from research_agent.tools import default_tool_registry

## Inspect the reusable pieces

The tool registry contains ordinary Python functions plus descriptions and argument schemas. The provider makes model requests. The runner owns the loop.

In [ ]:
tools = default_tool_registry()
for name, tool in tools.items():
    print(name, "→", tool.definition.description)

## Run a question requiring two tools

The provider reads the issued OpenRouter key from `.env`/the environment. The runner stops after at most five model turns.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

runner = AgentRunner(
    provider=OpenRouterProvider(),
    tools=tools,
    max_steps=5,
)
result = runner.run("Explain an AI agent using the local notes and calculate 12 * 7.")
print(result.status, result.steps, result.error)

## Observe every message in the loop

In [ ]:
for index, message in enumerate(result.messages):
    requested = [call.name for call in message.tool_calls]
    print(f"{index:02d} role={message.role:9} tool_requests={requested}")
    if message.role == "tool":
        print("   observation:", message.content[:160])

In [ ]:
print(result.response.model_dump_json(indent=2) if result.response else result.error)
print("Usage:", result.usage.model_dump())

## Read the control flow

Open `src/research_agent/agent.py` and identify: model call, final-answer validation, tool lookup, argument validation, duplicate-call check, and maximum-step termination.

The model proposes the next action. The application decides whether and how it is executed.

## Break it safely

Run with `max_steps=1`. Then request both explanation and calculation. Observe the explicit `max_steps` status instead of allowing an unbounded loop.

In [ ]:
limited = AgentRunner(OpenRouterProvider(), tools, max_steps=1)
limited_result = limited.run("Explain an AI agent using notes and calculate 12 * 7.")
print(limited_result.status, limited_result.error)

## Exercise and behaviour checks

Test: one direct question, one calculator question, one notes question, and one two-tool question. Record expected tool, actual tools, final schema validity, status, and steps.

The manual loop is now understandable but increasingly difficult to visualize and extend. Next we express the same mechanism as a graph—without replacing provider calls with LangChain abstractions.

## Your turn

Set max_steps to one and explain the incomplete trace.

## Recap

An agent loop is bounded application control around model decisions.
